<br>
<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%; background-color: white; margin-top: 50px;">
    <img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/>
    </div>
</a>
<h1 style="line-height: 1.4;"><font color="#76b900"><b>How to Run AI-Powered Computer-Aided Engineering Simulations</b></font></h1>
<h2><b>Notebook 1:</b> Crash Simulation Data Curation (ETL Pipeline)</h2>
<br>

### **From OpenRadioss D3plot to Zarr for AI Training**

**Target ML Frameworks:** NVIDIA PhysicsNeMo

---

This notebook guides you through the [**PhysicsNeMo-Curator**](https://github.com/NVIDIA/physicsnemo-curator/tree/main) pipeline. It demonstrates a **configuration-driven workflow** (via Hydra), allowing you to orchestrate complex data transformations using YAML files rather than writing raw Python scripts for every step.

---

## **1. Introduction**

This notebook demonstrates a unified **ETL (Extract, Transform, Load) pipeline** for Automotive Crash Dynamics simulation data.

The goal is to convert raw industrial simulation files from **OpenRadioss** (`d3plot` results + `*_0000.rad` input decks) into an **AI-optimized format (Zarr)** suitable for training deep learning surrogate models.

The dataset has been generated using the [OpenRadioss bumper beam example](https://openradioss.atlassian.net/wiki/spaces/OPENRADIOSS/pages/11075585/Bumper+Beam).


---

### Target AI Model

| Model | Architecture | How It Uses This Data |
|:------|:-------------|:----------------------|
| **GeoTransolver** | Geometry-Aware Transformer | Uses **geometric attention** to capture spatial relationships in the mesh structure. Processes **node positions**, **thickness features**, and **initial & boundary conditions** to predict crash dynamics. |



---

### The Workflow

<div style="display: flex; flex-direction: row;"><div style="float: left;">

| **INPUT** | | **CURATOR (This Notebook)** | | **OUTPUT** | **DESCRIPTION** |
|:----------|:---:|:----------------------------|:---:|:-----------|:----------------|
| `d3plot` (Binary Results) | ➜ | • Parse mesh & displacements | ➜ | **Zarr Dataset** | |
| `*_0000.rad` (Properties) | | • Remap node indices | | • `mesh_pos` $[T,N,3]$ | Absolute node positions over time (Reference + Displacement) |
| `runX.json` (Inital & Boundary Conditions) | | • Build graph edges | | • `thickness` $[N]$ | Per-node shell thickness from `/PROP/SHELL` definitions |
| | | • Map element → node data | | • `edges` $[E,2]$ | Graph connectivity derived from shell element topology |
| | | • Extract plastic strain & stress | | • `plastic_strain` $[T,N]$ | Effective plastic strain at each node (mapped from elements) |
| | | • Extract boundary conditions | | • `von_mises_stress` $[T,N]$ | Von Mises stress at each node (mapped from elements) |
| | | • Compress & chunk data | | • `velocity_vector` $[3]$ | Impact velocity vector $(v_x, v_y, v_z)$ in m/s from `runX.json` |
| | | | | • `rwall_diameter` $[1]$ | Rigid wall cylinder diameter in mm from `runX.json` |
| | | | | • `rwall_origin` $[3]$ | Rigid wall center position $(x, y, z)$ in mm from `runX.json` |

</div></div>

<br>


In [ ]:
from pathlib import Path
from utils.readers import preview_simulation_files

FEA_DATA_DIR = "/workspace/data/raw_data/"

# --- Define file paths and call the function ---
example_rad_file_path = FEA_DATA_DIR / "run1" / "Bumper_Beam_AP_meshed_0000.rad"
example_runjson_path = FEA_DATA_DIR / "run1" / "run1.json"

# You can easily adjust the start_line and num_lines right here
preview_simulation_files(example_rad_file_path, example_runjson_path, start_line=45, num_lines=30)

## **2. Installation and Usage**

**PhysicsNeMo-Curator** is a specialized submodule of the [PhysicsNeMo framework](https://github.com/NVIDIA/physicsnemo) designed to accelerate data curation using GPUs.

### **Recommended Environment: Docker**

The most robust way to run this pipeline is by leveraging the official **PhysicsNeMo Docker image** from the NVIDIA Container Registry (NGC). This ensures all complex dependencies (like `lasso`, `zarr`, and GPU libraries) are pre-configured.

**Step 1: Pull the Docker Image**
```bash
docker pull nvcr.io/nvidia/physicsnemo/physicsnemo:25.11
```
**Step 2: Run and Log into the Container**
```bash
docker run --gpus all -it --rm \
  -v /path/to/your/data:/data \
  -p 8888:8888 \
  --shm-size=1g \
  --ulimit memlock=-1 \
  --ulimit stack=67108864 \
  nvcr.io/nvidia/physicsnemo/physicsnemo:25.11 bash
```
**Step 3: Clone and Install the Curator**
```bash
# Install from source
git clone git@github.com:NVIDIA/physicsnemo-curator.git && cd physicsnemo-curator

pip install --upgrade pip
pip install -e ".[dev]"

# Install pre-commit hooks
pre-commit install
```


---

## **3. Deep Inspection of D3plot, RAD, and JSON Files**

The following code performs a comprehensive analysis of our crash simulation data.

In [ ]:
from utils.d3plot_utils import inspect_and_plot_d3plot

# --- CONFIGURATION ---
d3plot_path = FEA_DATA_DIR / "run1" / "d3plot"

inspect_and_plot_d3plot(d3plot_path)

---

## **4. Interpreting the Results**

<div style="display: flex; flex-direction: row;"><div style="float: left;">

### 1. Simulation Overview

The data summary reveals the scale and scope of this crash simulation:

| Metric | Value | Interpretation |
|:-------|:------|:---------------|
| **Time Steps** | 51 frames | Captures the crash event from start to finish |
| **Simulation Time** | 0 → 2 s | Full crash duration (~40 ms intervals between frames) |
| **Nodes** | 13,882 | The "atoms" of the mesh — points where physics is computed |
| **Dimensions** | (51, 13882, 3) | 51 timesteps × 13,882 nodes × 3 coordinates (X, Y, Z) |

### 2. Coordinate System Detection

The script detected that displacement values at T=0 were **~2045 mm** — these are **absolute world coordinates**, not relative displacements. 

The fix subtracts the initial position to obtain true deformation:

<div style="display: flex; flex-direction: row;"><div style="float: left;">

> $\Large \mathbf{u}(t) = \mathbf{x}(t) - \mathbf{x}(0)$

Where:
- $\mathbf{x}(t)$ = absolute position at time $t$
- $\mathbf{x}(0)$ = initial position  
- $\mathbf{u}(t)$ = true displacement (what we want for ML training)


### 3. Deformation Results

| Metric | Value |
|:-------|:------|
| **Max Deformation** | 264.28 mm |

This is the **peak displacement magnitude** across all 13,882 nodes over the entire simulation. 

### 4. Thickness Analysis

The thickness data was extracted from the OpenRadioss input file (`*_0000.rad`):

<div style="display: flex; flex-direction: row;"><div style="float: left;">

| Property ID | Thickness (mm) | Likely Component |
|:------------|:---------------|:-----------------|
| 1 | 2.200 | Main beam structure |
| 2 | 1.800 | Reinforcement / brackets |
| 6 | 2.200 | Main beam structure |
| 7 | 1.800 | Reinforcement / brackets |

### 5. Training Variable Distributions (Histograms)

The histograms above visualize the **statistical distributions** of each variable that will be used for model training:

| Histogram | What It Shows | Key Observations |
|:----------|:--------------|:-----------------|
| **Node Displacement** | Distribution of displacement magnitudes at the final timestep | most nodes either moved linearly or hit the rigid pole and remained there (~200mm in crash zone) |
| **Shell Thickness** | Distribution of material thickness values | Two discrete values (1.8mm and 2.2mm) — indicates a simple design with two part types |
| **Plastic Strain** | Distribution of permanent deformation across nodes | Heavily right-skewed — most material remains elastic, only the impact zone shows significant plastic deformation |
| **Von Mises Stress** | Distribution of equivalent stress intensity | Right-skewed with peak near zero — residual stress after unloading is concentrated in deformed regions |

#### Why These Distributions Matter for Physics AI

Understanding training data distributions is **critical** for building robust surrogate models:

| Insight | Implication for Model Training |
|:--------|:-------------------------------|
| **Skewed distributions** (strain, stress) | May require **log-transform** or **quantile normalization** to prevent the model from ignoring rare but important high-value regions |
| **Long tails** in stress/strain | High-value outliers represent the most critical physics (yield, failure) — ensure loss function doesn't underweight these samples |

> **Best Practice**: Before training, always visualize your feature distributions. Normalize features to similar scales (e.g., zero mean, unit variance) and consider class balancing for imbalanced physical phenomena like plastic strain localization.

---
## **5. The PhysicsNeMo Curator Processing Pipeline**

The ETL (Extract-Transform-Load) pipeline consists of three distinct stages designed to turn raw physics simulation data into clean, ML-ready features.


```mermaid
graph TD
    subgraph Extract [<b>1 - Source</b>]
        A1[<b>d3plot</b><br>Nodal Pos, Stress, Strain] --> B(Parser)
        A2[<b>*_0000.rad</b><br>Connectivity & Properties] --> B
        A3[<b>runX.json</b><br>Boundary Conditions] --> B
    end

    subgraph Transform [<b>2 - Transformation</b>]
        B -- Raw Arrays --> D[<b>Spatial Mapping</b><br>mesh_pos]
        D --> E[<b>Attribute Mapping</b><br>plastic_strain / stress]
        E --> F[<b>Topological Mapping</b><br>thickness / edges]
        F --> G[<b>BC Extraction</b><br>velocity / wall params]
    end

    subgraph Load [<b>3 - Sink</b>]
        G --> H[<b>Chunk & Compress</b>]
        H --> I[(<b>dataset.zarr/</b>)]
    end

    style Extract fill:#e1f5fe,stroke:#01579b
    style Transform fill:#fff3e0,stroke:#e65100
    style Load fill:#e8f5e9,stroke:#1b5e20
```

### **Modular Architecture: Code Meets Configuration**

The pipeline diagram above maps directly to the **PhysicsNeMo-Curator** library structure. Each stage is implemented as a modular Python component, orchestrated by a **Hydra YAML configuration file**:

```
physicsnemo-curator/examples/structural_mechanics/crash/
├── config/
│   ├── crash_etl.yaml              # Main pipeline orchestration
│   └── serialization_format/
│       ├── zarr.yaml               # Zarr output settings
│       └── vtp.yaml                # VTP output settings
├── data_sources.py                 # SOURCE: CrashD3PlotDataSource, CrashZarrDataSource
├── data_transformations.py         # TRANSFORM: CrashDataTransformation
├── crash_data_processors.py        # PROCESSORS: Helper functions (node filtering, edge building)
├── schemas.py                      # Data contracts and type definitions
└── run_etl.py                      # Entry point
```

#### **The Configuration File: `crash_etl.yaml`**

This YAML file defines *what* components to use and *how* to connect them:

```yaml
defaults:
  - /serialization_format: zarr      # Default output format
  - _self_

etl:
  processing:
    num_processes: 12               # Parallel workers for multi-run processing

  source:                           # STAGE 1: Where data comes from
    _target_: data_sources.CrashD3PlotDataSource
    input_dir: ???                  # Provided via CLI

  transformations:                  # STAGE 2: How data is processed
    crash_transform:
      _target_: data_transformations.CrashDataTransformation
      wall_threshold: 1.0           # Filter nodes with displacement < 1mm

  sink: ${serialization_format.sink}  # STAGE 3: Where data goes (from zarr.yaml or vtp.yaml)
```

| YAML Section | Maps To | Python Component |
|:-------------|:--------|:-----------------|
| `etl.source` | **Stage 1 (Source)** | `data_sources.CrashD3PlotDataSource` |
| `etl.transformations` | **Stage 2 (Transform)** | `data_transformations.CrashDataTransformation` |
| `etl.sink` | **Stage 3 (Sink)** | `data_sources.CrashZarrDataSource` or `CrashVTPDataSource` |

Now let's explore each stage in detail:

---

### **Stage 1: Data Extraction (Source)**

<div style="display: flex; flex-direction: row;"><div style="float: left;">

| Property | Description |
|:---------|:------------|
| **Component** | `CrashD3PlotDataSource` |
| **Function** | Handles parsing of binary simulation results (`d3plot`), OpenRadioss input decks (`*_0000.rad`), and boundary condition parameters (`runX.json`) |

> ⚠️ **FEA vs. ML Data Representation**: Finite Element Analysis (FEA) solvers compute physical quantities like **stress**, **strain**, and **thickness** at the **element level** (shell/solid centers). However, GeoTransolver operates on **nodes** (graph vertices). This fundamental mismatch means we must **map element-based variables to nodes** — this mapping happens here in the Source stage.
>
> $$\text{node\_value}_i = \frac{1}{|\mathcal{E}_i|} \sum_{e \in \mathcal{E}_i} \text{element\_value}_e$$ where $\mathcal{E}_i$ is the set of all elements connected to node $i$.

### Von Mises Stress Computation

For stress, we first compute the **von Mises equivalent stress** at element centers:

$$\sigma_{VM} = \sqrt{\frac{1}{2}\left[(\sigma_{xx} - \sigma_{yy})^2 + (\sigma_{yy} - \sigma_{zz})^2 + (\sigma_{zz} - \sigma_{xx})^2 + 6\sigma_{xy}^2\right]}$$

Then map this scalar to nodes using the averaging method above.

In [ ]:
import sys
from pathlib import Path
from utils.readers import print_record_info
from utils.d3plot_utils import *

# Ensure physicsnemo-curator and crash directory are in sys.path
project_root = "workspace/physicsnemo-curator"
crash_dir = project_root / "examples" / "structural_mechanics" / "crash"
for path in [project_root, crash_dir]:
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from examples.structural_mechanics.crash.data_sources import *
from examples.structural_mechanics.crash.data_transformations import *

import numpy as np
from lasso.dyna import D3plot, ArrayType
import json

class CrashD3PlotDataSource(DataSource):
    """Data source for reading LS-DYNA d3plot simulation files."""

    def __init__(
        self,
        cfg: ProcessingConfig,
        input_dir: str,
        boundary_condition_keys: Optional[List[str]] = None,
    ):
        """Initialize the D3plot data source.

        Args:
            cfg: Processing configuration.
            input_dir: Directory containing run folders with d3plot files.
            boundary_condition_keys: List of parameter keys to extract from run JSON files
                as boundary conditions (e.g., ["velocity_vector", "rwall_diameter", "rwall_origin"]).
                If None, no boundary conditions are extracted.
        """
        super().__init__(cfg)
        self.input_dir = Path(input_dir)
        self.boundary_condition_keys = boundary_condition_keys or []
        self.logger = logging.getLogger(__name__)
        ...

    def get_file_list(self) -> List[str]:
        """Find all run folders containing d3plot files."""
        run_folders = []
        for item in self.input_dir.iterdir():
            if item.is_dir():
                d3plot_path = item / "d3plot"
                if d3plot_path.exists():
                    run_folders.append(item.name)

    def read_file(self, run_id: str) -> CrashExtractedDataInMemory:
        """Read d3plot, .k/.rad file, and JSON boundary conditions for one simulation run.

        Returns CrashExtractedDataInMemory object.
        """
        run_dir = self.input_dir / run_id
        d3plot_path = run_dir / "d3plot"

        # ✅ Node coordinates and temporal displacements from `d3plot`
        # ✅ Shell element connectivity (triangles/quads) and Part IDs
        # ✅ **Element→Node Projection**: Plastic strain & von Mises stress mapped from element centers to nodes via weighted averaging
        (
            coords,
            pos_raw,
            mesh_connectivity,
            part_ids,
            actual_part_ids,
            plastic_strain,
            von_mises_stress,
        ) = load_d3plot_data(str(d3plot_path))
        # ✅ load_d3plot_data is a utility tool from physicsnemo-curator.

        # Initialize defaults
        node_thickness = np.zeros(len(coords))

        # Find metadata files
        rad_file_path = find_rad_file(run_dir)
        
        #✅ **Thickness Mapping**: Part IDs → `/PROP/SHELL` → Nodes (averaged if node spans multiple parts)
        part_thickness_map = {}
        part_thickness_map = parse_rad_file(rad_file_path=rad_file_path)
        node_thickness = compute_node_thickness(
            mesh_connectivity=mesh_connectivity,
            part_ids=part_ids,
            part_thickness_map=part_thickness_map,
            actual_part_ids=actual_part_ids,
            num_nodes=len(coords),
        )

        # ✅ Boundary conditions (velocity vector, rigid wall geometry) from `runX.json`
        boundary_conditions = self._read_boundary_conditions(run_dir, run_id)

        return CrashExtractedDataInMemory(
            metadata=CrashMetadata(
                filename=run_id,
            ),
            pos_raw=pos_raw,
            mesh_connectivity=mesh_connectivity,
            node_thickness=node_thickness,
            node_plastic_strain=plastic_strain,
            node_von_mises_stress=von_mises_stress,
        )

    def _read_boundary_conditions(self, run_dir: Path, run_id: str) -> Dict[str, Any]:
        ...
        return boundary_conditions

In [ ]:
from examples.structural_mechanics.crash.data_sources import CrashD3PlotDataSource

## Example Execution
data_source = CrashD3PlotDataSource(
    cfg = ProcessingConfig(num_processes=8),
    input_dir = FEA_DATA_DIR,
    boundary_condition_keys = ["velocity_vector", "rwall_diameter", "rwall_origin"],
)
run1_record = data_source.read_file("run1")
    
print_record_info(run1_record, "run1_record")

### Validation

You can verify the mapping quality by:
- ✅ Checking numpy array shapoes match expecteded sizes.
- ✅ Checking node value ranges are within element value bounds.


---

### **Stage 2: Data Transformation**

<div style="display: flex; flex-direction: row;"><div style="float: left;">

| Property | Description |
|:---------|:------------|
| **Component** | `CrashDataTransformation` |
| **Function** | Applies sequential transformations to prepare data for ML |

</div></div>

**Transformation Steps:**

<div style="display: flex; flex-direction: row;"><div style="float: left;">

| Order | Step | Description |
|:------|:-----|:------------|
| 1 | **Wall Node Filtering** | Removes rigid/static nodes with displacement below threshold (e.g., `< 1.0 mm`). These nodes represent contact surfaces or boundary constraints that don't deform during the crash — keeping them would not add information to the training data. |
| 2 | **Array Filtering** | Applies the wall node mask to all arrays: positions, thickness, plastic strain, von Mises stress |
| 3 | **Connectivity Remapping** | Updates element connectivity to reference the new (filtered) node indices |
| 4 | **Orphan Node Compaction** | Removes nodes not referenced by any element and reindexes to contiguous $0 \to N-1$ |

---



In [ ]:
class CrashDataTransformation(DataTransformation):
    """Transform crash simulation data: filter nodes, compute thickness, build edges."""

    def __init__(
        self,
        cfg: ProcessingConfig,
        wall_threshold: float = 1.0,
        compact_orphan_nodes: bool = True,
        crash_processors: Optional[tuple[Callable, ...]] = None,
    ):
        """Initialize the crash data transformation.

        Args:
            cfg: Processing configuration.
            wall_threshold: Max displacement below which a node is considered "wall".
            compact_orphan_nodes: If True, remove nodes not referenced by any mesh cell
                and reindex to contiguous 0..N-1. If False (default), preserve all nodes
                including unconnected ones (e.g., wall/contact surface nodes).
            crash_processors: Optional tuple of additional processor functions.
        """
        super().__init__(cfg)
        self.logger = logging.getLogger(__name__)
        self.wall_threshold = wall_threshold
        self.compact_orphan_nodes = compact_orphan_nodes
        self.crash_processors = crash_processors

    def transform(
        self,
        data: CrashExtractedDataInMemory,
    ) -> CrashExtractedDataInMemory:
        """Transform raw d3plot data into VTP format.

        Steps:
        1. Identify wall nodes (low displacement)
        2. Filter arrays
        3. Remap mesh connectivity
        4. Verify cells remain after filtering
        5. Optionally compact orphan nodes (if compact_orphan_nodes=True)

        Returns dict with:
        - filtered_pos_raw: (timesteps, filtered_nodes, 3)
        - filtered_mesh_connectivity: remapped connectivity
        - node_thickness: per-node thickness values
        """
        
        # Step 1: Identify wall nodes
        node_type = compute_node_type(data.pos_raw, threshold=self.wall_threshold)
        keep_nodes = sorted(np.where(node_type == 0)[0])  # keep structure
        node_map = {old_idx: new_idx for new_idx, old_idx in enumerate(keep_nodes)}

        # Step 2: Filter arrays
        filtered_pos_raw = data.pos_raw[:, keep_nodes, :]
        filtered_node_thickness = data.node_thickness[keep_nodes]

        # Filter plastic strain and stress:
        filtered_plastic_strain = None
        filtered_plastic_strain = data.node_plastic_strain[:, keep_nodes]

        filtered_von_mises_stress = None
        filtered_von_mises_stress = data.node_von_mises_stress[:, keep_nodes]
        
        # Step 3: Remap mesh connectivity
        filtered_mesh_connectivity = []
        for cell in data.mesh_connectivity:
            filtered_cell = [node_map[n] for n in cell if n in node_map]
            if len(filtered_cell) >= 3:
                filtered_mesh_connectivity.append(filtered_cell)

        # Step 4: Verify we have cells remaining
        used = np.unique(
            np.array([i for cell in filtered_mesh_connectivity for i in cell])
        )
        if used.size == 0:
            raise ValueError("No cells left after filtering")

        return CrashExtractedDataInMemory(
            metadata=data.metadata,
            filtered_pos_raw=filtered_pos_raw,
            filtered_mesh_connectivity=filtered_mesh_connectivity,
            filtered_node_thickness=filtered_node_thickness,
            filtered_plastic_strain=filtered_plastic_strain,
            filtered_von_mises_stress=filtered_von_mises_stress
        )  

In [ ]:
from examples.structural_mechanics.crash.data_transformations import CrashDataTransformation

transformation = CrashDataTransformation(cfg = ProcessingConfig(num_processes=8))
run1_record_tf = transformation.transform(data = run1_record)

print_record_info(run1_record_tf, "run1_record_tf")


### **Stage 3: Data Loading (Sink)**

Zarr is an open source project to develop specifications and software for storage of large N-dimensional typed arrays.
Zarr provides support for storage using distributed systems like cloud object stores,
and enables efficient I/O for parallel computing applications.

Key features relevant for AI model training:

- **Chunk multi-dimensional arrays** along any dimension
- **Store arrays** in memory, on disk, inside a Zip file, on S3, etc.
- **Read and write arrays concurrently** from multiple threads or processes
- **Organize arrays into hierarchies** via annotatable groups
- **Built-in compression** reduces storage requirements while maintaining fast access
- **Language agnostic**: Accessible from Python, C, C++, Rust, Javascript, Java, and other frameworks

For more information, see the [Zarr project website](https://zarr.dev/).

For our use case, Zarr is especially useful because it lets our data loader read only *relevant* time steps (e.g., $t$ = 0.05s) on demand, instead of loading the full simulation into memory.

<div style="display: flex; flex-direction: row;"><div style="float: left;">

| Property | Description |
|:---------|:------------|
| **Component** | `CrashZarrDataSource` |
| **Function** | Writes processed data to disk in Zarr format with optimized chunking and compression |

**Storage Optimizations:**

| Feature | Configuration | Benefit |
|:--------|:--------------|:--------|
| **Chunking** | Adaptive (~1.0 MB per chunk) | Enables efficient parallel I/O and random access to timesteps |
| **Compression** | Blosc/zstd (Level 3) | ~2-3x storage reduction while maintaining fast decompression |
| **Atomic Writes** | Temp file → rename | Prevents corrupted data from interrupted runs |

**Chunking Strategy for Temporal Arrays:**

For 3D arrays like `mesh_pos` $[T, N, 3]$, the chunking prioritizes **timestep access**:

1. Calculate how many timesteps fit in ~1 MB
2. Keep spatial coordinates (3 for XYZ) intact
3. Result: chunks of `(chunk_timesteps, N, 3)`

**Example** with shape `[51, 13675, 3]` (float32):
- One timestep = $13,675 \times 3 \times 4$ bytes $\approx$ **0.16 MB**
- Target 1 MB → **~6 timesteps per chunk**
- Final chunks: `(6, 13675, 3)`

This enables PyTorch DataLoaders to fetch only the timesteps needed for each training batch — no need to load the entire simulation into memory.

In [ ]:
class CrashZarrDataSource(DataSource):
    """Data source for writing crash simulation data to Zarr format."""

    def __init__(
        self,
        cfg: ProcessingConfig,
        output_dir: str,
        overwrite_existing: bool = True,
        compression_level: int = 3,
        compression_method: str = "zstd",
        chunk_size_mb: float = 1.0,
    ):
        """Initialize the Zarr data source.

        Args:
            cfg: Processing configuration
            output_dir: Directory to write Zarr stores
            overwrite_existing: Whether to overwrite existing files
            compression_level: Compression level (1-9, higher = more compression)
            compression_method: Compression method
            chunk_size_mb: Target chunk size in MB (default: 1.0)
        """
        super().__init__(cfg)
        self.output_dir = Path(output_dir)
        self.overwrite_existing = overwrite_existing
        self.compression_level = compression_level
        self.compression_method = compression_method
        self.chunk_size_mb = chunk_size_mb

        # Set up compressor
        self.compressor = zarr.codecs.BloscCodec(
            cname=compression_method,
            clevel=compression_level,
            shuffle=zarr.codecs.BloscShuffle.shuffle,
        )

        self.output_dir.mkdir(parents=True, exist_ok=True)

    def _calculate_chunks(self, array: np.ndarray) -> tuple:
        """Calculate optimal chunk sizes based on target chunk size in MB.

        Args:
            array: Array to calculate chunks for

        Returns:
            Tuple of chunk dimensions
        """
        target_chunk_size = int(self.chunk_size_mb * 1024 * 1024)  # Convert MB to bytes
        item_size = array.itemsize
        shape = array.shape

        # 3D array (e.g., mesh_pos with shape [T, N, 3]):
        # Try to balance between timesteps and nodes
        # Keep the last dimension (3 for coordinates) intact
        elements_per_slice = shape[1] * shape[2]
        chunk_timesteps = max(
            1, min(shape[0], target_chunk_size // (item_size * elements_per_slice))
        )

        # If we can fit multiple timesteps, reduce node chunks
        if chunk_timesteps >= shape[0]:
            # All timesteps fit, chunk along nodes
            chunk_nodes = min(
                shape[1],
                max(1, target_chunk_size // (item_size * shape[0] * shape[2])),
            )
            return (shape[0], max(1, chunk_nodes), shape[2])
        else:
            # Chunk along timesteps, keep reasonable node chunks
            remaining_size = target_chunk_size // (
                item_size * chunk_timesteps * shape[2]
            )
            chunk_nodes = min(shape[1], max(1, remaining_size))
            return (chunk_timesteps, max(1, chunk_nodes), shape[2])

    def _get_output_path(self, filename: str) -> Path:
        return self.output_dir / f"{filename}.zarr"

    def _write_impl_temp_file(
        self, data: CrashExtractedDataInMemory, output_path: Path
    ) -> None:
        """Write crash data to a Zarr store.

        This method receives a TEMPORARY path from the base class (e.g., run_001.zarr_temp).
        After writing completes, the base class atomically renames it to the final path.

        Args:
            data: Transformed crash data containing:
                - filtered_pos_raw: (timesteps, nodes, 3) temporal positions
                - filtered_mesh_connectivity: list of cell connectivities
                - filtered_node_thickness: (nodes,) thickness values
                - edges: (num_edges, 2) edge connectivity
                - metadata.boundary_conditions: dict of boundary condition values
            output_path: TEMPORARY directory path for the Zarr store.
                        Base class will rename this to final path after writing completes.
        """
        self.logger.info(
            f"Creating Zarr store at temporary location: {output_path.name}"
        )

        # Create Zarr store
        zarr_store = LocalStore(output_path)
        root = zarr.group(store=zarr_store)

        # Write metadata as root attributes
        root.attrs["filename"] = data.metadata.filename
        root.attrs["num_timesteps"] = data.filtered_pos_raw.shape[0]
        root.attrs["num_nodes"] = data.filtered_pos_raw.shape[1]
        root.attrs["num_edges"] = len(data.edges)
        root.attrs["compression"] = self.compression_method
        root.attrs["compression_level"] = self.compression_level
        root.attrs["chunk_size_mb"] = self.chunk_size_mb

        # Convert data to appropriate dtypes
        num_timesteps, num_nodes, _ = data.filtered_pos_raw.shape
        mesh_pos_data = data.filtered_pos_raw.astype(np.float32)
        thickness_data = data.filtered_node_thickness.astype(np.float32)
        edges_array = np.array(list(data.edges), dtype=np.int64)

        # Calculate optimal chunks for each array
        mesh_pos_chunks = self._calculate_chunks(mesh_pos_data)
        thickness_chunks = self._calculate_chunks(thickness_data)
        edges_chunks = self._calculate_chunks(edges_array)

        # Write temporal position data
        root.create_array(
            name="mesh_pos",
            data=mesh_pos_data,
            chunks=mesh_pos_chunks,
            compressors=(self.compressor,),
        )

        # Write node thickness (static per node)
        root.create_array(
            name="thickness",
            data=thickness_data,
            chunks=thickness_chunks,
            compressors=(self.compressor,),
        )

        # Write edges connectivity
        root.create_array(
            name="edges",
            data=edges_array,
            chunks=edges_chunks,
            compressors=(self.compressor,),
        )

        # Write plastic strain if available
        plastic_strain_data = data.filtered_plastic_strain.astype(np.float32)
        plastic_strain_chunks = self._calculate_chunks(plastic_strain_data)
        root.create_array(
            name="plastic_strain",
            data=plastic_strain_data,
            chunks=plastic_strain_chunks,
            compressors=(self.compressor,),
        )
        root.attrs["plastic_strain_min"] = float(np.min(plastic_strain_data))
        root.attrs["plastic_strain_max"] = float(np.max(plastic_strain_data))
        root.attrs["plastic_strain_mean"] = float(np.mean(plastic_strain_data))

        # Write von Mises stress if available
        stress_data = data.filtered_von_mises_stress.astype(np.float32)
        stress_chunks = self._calculate_chunks(stress_data)
        root.create_array(
            name="von_mises_stress",
            data=stress_data,
            chunks=stress_chunks,
            compressors=(self.compressor,),
        )
        root.attrs["von_mises_stress_min"] = float(np.min(stress_data))
        root.attrs["von_mises_stress_max"] = float(np.max(stress_data))
        root.attrs["von_mises_stress_mean"] = float(np.mean(stress_data))

        # Add some statistics as metadata
        root.attrs["thickness_min"] = float(np.min(data.filtered_node_thickness))
        root.attrs["thickness_max"] = float(np.max(data.filtered_node_thickness))
        root.attrs["thickness_mean"] = float(np.mean(data.filtered_node_thickness))

        # Write boundary conditions as both attributes and arrays
        # Create a boundary_conditions group for organized access
        bc_group = root.create_group("boundary_conditions")

        for key, value in data.metadata.boundary_conditions.items():
            # Store as attribute (for quick access to metadata)
            root.attrs[f"bc_{key}"] = value

            # Also store as array in the boundary_conditions group
            # This allows consistent array-based access in training
            if isinstance(value, (list, tuple)):
                bc_array = np.array(value, dtype=np.float64)
            elif isinstance(value, (int, float)):
                bc_array = np.array([value], dtype=np.float64)
            else:
                self.logger.warning(
                    f"Skipping boundary condition '{key}' with unsupported type: {type(value)}"
                )
                continue

            bc_group.create_array(
                name=key,
                data=bc_array,
                chunks=bc_array.shape,  # Small arrays, no chunking needed
            )

In [ ]:
import tempfile
import zarr
import math
from pathlib import Path
from examples.structural_mechanics.crash.data_sources import CrashZarrDataSource
# Assuming ProcessingConfig and run1_record_tf are defined elsewhere

with tempfile.TemporaryDirectory() as temp_dir:
    sink = CrashZarrDataSource(ProcessingConfig(num_processes=8), temp_dir, overwrite_existing=True)
    sink.write(run1_record_tf, "test_run")

    # Check store was created
    output_store = Path(temp_dir) / "test_run.zarr"

    # Open and verify the Zarr store
    store = zarr.open(str(output_store), mode="r")

    print("-" * 115)
    print(f"{'Key':<20} | {'Shape':<15} | {'Chunks':<15} | {'Dtype':<8} | {'Total Size':<12} | {'Chunk Size'}")
    print("-" * 115)
    
    for key, array in store.arrays():
        shape = str(array.shape)
        chunks = str(array.chunks)
        dtype = str(array.dtype)
        
        # Calculate sizes in MB
        total_size_mb = array.nbytes / (1024 * 1024)
        
        # Calculate the size of a single chunk:
        chunk_size_mb = (math.prod(array.chunks) * array.dtype.itemsize) / (1024 * 1024)
        print(f"{key:<20} | {shape:<15} | {chunks:<15} | {dtype:<8} | {total_size_mb:>7.3f} MB | {chunk_size_mb:>7.3f} MB")
    print("-" * 115)

---
## **6. Configuration Options (YAML)**
We do not modify Python code to run the ETL. We use Hydra configuration files located in config.

### A. Main Pipeline Config: `crash_etl.yaml`
This file (**[`examples/structural_mechanics/crash/config/crash_etl.yaml`](https://github.com/NVIDIA/physicsnemo-curator/blob/main/examples/structural_mechanics/crash/config/crash_etl.yaml)**) controls what happens to the data.

```yaml
defaults:
  - /serialization_format: zarr  # Default format (we will override this to 'zarr' later)
  - _self_

etl:
  processing:
    num_processes: 12  # PARALLELISM: Processes multiple "RunXXX" folders at once.

  source:
    _target_: data_sources.CrashD3PlotDataSource
    _convert_: all
    input_dir: ???     # Placeholder: We will provide the RAW_DATA path in the CLI

  transformations:
    crash_transform:
      _target_: data_transformations.CrashDataTransformation
      _convert_: all
      
      # PHYSICS PARAMETER: Wall Threshold
      # Nodes moving less than 1.0 unit (mm) during the crash are removed.
      # This cleans the data of rigid parts and reduces file size by ~40%.
      wall_threshold: 1.0 

  sink: ${serialization_format.sink}
```

### B. Output Config: `zarr.yaml`

This file (**[`config/serialization_format/zarr.yaml`](./config/serialization_format/zarr.yaml)**) controls the **Output Format**.

```yaml
sink:
  _target_: data_sources.CrashZarrDataSource
  _convert_: all
  output_dir: ???           # Placeholder for output path
  overwrite_existing: true
  
  # Compression: Reduces storage size (Zstd is fast and efficient)
  compression_level: 3
  compression_method: "zstd"
  
  # Chunking: Critical for training performance.
  # 1.0 MB is optimized for streaming data to the GPU.
  chunk_size_mb: 1.0
```


---
## 7. Running the Pipeline (CLI)
Now we execute the pipeline. We will override the configuration placeholders to point to your specific directories.

**Command Breakdown:**

1. `etl.source.input_dir`: Points to your raw data.
2. `serialization_format=zarr`: Tells the system to use the Zarr config.
3. `serialization_format.sink.output_dir`: Where to save the clean data.

In [ ]:
!python physicsnemo-curator/examples/structural_mechanics/crash/run_etl.py \
    --config-dir=configs \
    --config-name=crash_etl \
    serialization_format=zarr \
    etl.source.input_dir=$FEA_DATA_DIR \
    serialization_format.sink.output_dir=/workspace/processed_data_zarr \
    serialization_format.sink.compression_level=5 \
    etl.processing.num_processes=8

As a sanity check, the data you just generated should match the data we have vendored in `/data/processed_data_local`:

In [ ]:
print("Deviation From Vendored (If Any):")
!diff -r /workspace/processed_data_zarr/run2.zarr /data/processed_data_local/run2.zarr
print("\nComputed Structure:")
!du -h /workspace/processed_data_zarr/run2.zarr

---
## 8. Verifying the Zarr Output
Once the command finishes, check the output folder. You should see .zarr directories corresponding to your input runs.
Let's inspect the processed data to ensure the Data Contract is met (Nodes, Edges, Thickness).

In [ ]:
import zarr
import os
import numpy as np

run_name = "run1"
output_zarr_path = Path.cwd() / "processed_data_zarr" / f"{run_name}.zarr"

if os.path.exists(output_zarr_path):
    # Open the Zarr group
    store = zarr.open_group(output_zarr_path, mode='r')
    
    print(f"--- Inspection of {output_zarr_path} ---")
    
    # 1. Mesh Position (mesh_pos)
    pos = store['mesh_pos']
    print(f"Positions Shape: {pos.shape} | Type: {pos.dtype}")
    
    # Calculate ranges for X, Y, and Z
    # Using np.min/max on the zarr object loads data into memory; 
    # for extremely large datasets, consider chunk-wise processing.
    pos_min = np.min(pos, axis=(0, 1))
    pos_max = np.max(pos, axis=(0, 1))
    
    print(f"  -> X Range: [{pos_min[0]:.4f}, {pos_max[0]:.4f}]")
    print(f"  -> Y Range: [{pos_min[1]:.4f}, {pos_max[1]:.4f}]")
    print(f"  -> Z Range: [{pos_min[2]:.4f}, {pos_max[2]:.4f}]")
    
    # 2. Thickness (thickness)
    thick = store['thickness']
    print(f"\nThickness Shape: {thick.shape} | Type: {thick.dtype}")
    
    thick_min = np.min(thick)
    thick_max = np.max(thick)
    print(f"  -> Value Range: [{thick_min:.4f}, {thick_max:.4f}]")
    
    # 3. Plastic Strain (plastic_strain) - if available
    if 'plastic_strain' in store:
        pstrain = store['plastic_strain']
        print(f"\nPlastic Strain Shape: {pstrain.shape} | Type: {pstrain.dtype}")
        pstrain_min = np.min(pstrain)
        pstrain_max = np.max(pstrain)
        print(f"  -> Value Range: [{pstrain_min:.6f}, {pstrain_max:.6f}]")
    else:
        print("\n⚠️ Plastic Strain not found (may not be available in this d3plot)")
    
    # 5. Von Mises Stress (von_mises_stress) - if available
    if 'von_mises_stress' in store:
        stress = store['von_mises_stress']
        print(f"\nVon Mises Stress Shape: {stress.shape} | Type: {stress.dtype}")
        stress_min = np.min(stress)
        stress_max = np.max(stress)
        print(f"  -> Value Range: [{stress_min:.2f}, {stress_max:.2f}] MPa")
    else:
        print("\n⚠️ Von Mises Stress not found (may not be available in this d3plot)")
    
    # 6. Boundary Conditions (from run JSON file)
    # Method 2: Access as arrays (consistent array-based access for training)
    print(f"\n--- Boundary Conditions ---")
    if "boundary_conditions" in store:
        bc_group = store["boundary_conditions"]
        bc_keys = list(bc_group.keys())
        print(f"Available BC keys: {bc_keys}")
        
        for key in bc_keys:
            bc_array = bc_group[key][:]
            print(f"  {key}: {bc_array} (shape: {bc_array.shape}, dtype: {bc_array.dtype})")
        
        # Example: Concatenate all BCs into a single feature vector for model conditioning
        bc_arrays = [bc_group[key][:] for key in bc_keys]
        bc_feature_vector = np.concatenate(bc_arrays)
        print(f"\n  Combined BC Feature Vector: {bc_feature_vector}")
        print(f"  Feature Vector Shape: {bc_feature_vector.shape}")
    else:
        print("  No boundary conditions found in this Zarr store.")
        print("  (Run the ETL with boundary_condition_keys configured to extract them)")
    
    # 5. Storage & Compression Details
    print(f"\n--- Storage Details (Zarr Info) ---")
    print(pos.info)

else:
    print(f"Output file for {run_name} not found.")
    print("Please ensure you ran the pipeline command in Section 5 successfully.")

## **9. Decoding the Zarr Output**

The output above confirms the pipeline was successful. Here's a detailed breakdown of what these numbers mean for AI training:

### 1. Performance Features (Chunking & Compression)

#### Chunking Configuration

<div style="display: flex; flex-direction: row;"><div style="float: left;">

| Aspect | Configuration | Benefit |
|:-------|:--------------|:--------|
| **Time chunking** | 6 timesteps per chunk | Load ~6 frames at a time |
| **Spatial chunking** | Full mesh per chunk | No fragmentation across nodes |
| **Chunk size** | ~1 MB | Optimized for GPU memory transfer |

**Why This Matters for Training:**

When the DataLoader needs timestep $t=15$, it reads only the chunk containing that frame — **not the entire simulation**. 
This enables:
-  Fast random access to any timestep
-  Memory-efficient streaming
-  Parallel I/O from multiple workers

#### Compression Settings

<div style="display: flex; flex-direction: row;"><div style="float: left;">

| Setting | Value | Effect |
|:--------|:------|:-------|
| **Algorithm** | `zstd` (Zstandard) | High-performance compression |
| **Level** | 5 | Balanced compression ratio vs. speed |
| **Shuffle** | Enabled | Reorders bytes for better compression of numerical data |

**Storage Efficiency:**

| Metric | Value |
|:-------|:------|
| **Uncompressed Size** | ~8.0 MB |
| **On-Disk Size** | ~3-4 MB (estimated) |
| **Compression Ratio** | ~2-3x |

### 3. Data Precision

| Original (Simulation) | Converted (ML) | Memory Savings |
|:----------------------|:---------------|:---------------|
| Float64 (8 bytes) | Float32 (4 bytes) | **50% reduction** |

**Why Float32?**

| Reason | Explanation |
|:-------|:------------|
| **GPU Optimization** | NVIDIA GPUs have dedicated FP32 Tensor Cores |
| **Memory Efficiency** | Fit larger batches in GPU VRAM |
| **Sufficient Precision** | ML training doesn't require Float64 precision |
| **Industry Standard** | PyTorch and TensorFlow default to FP32 |

</div></div>




## **10. Alternative Output: VTP Format (Visualization)**

While **Zarr** is optimized for high-speed AI training, it is opaque to humans. To visually inspect your data, check for "flying nodes," or validate the wall filtering, you should generate **VTP (Visualization Toolkit PolyData)** files.

* **Visual Debugging:** Open files directly in **ParaView** or **PyVista**.
* **Sanity Checks:** Verify that the mesh connectivity looks correct after transformation.

<div style="display: flex; flex-direction: row;"><div style="float: left;">

### **B. The VTP Data Structure**
Unlike Zarr (which stores absolute positions), VTP stores a **Reference Mesh** plus **Displacement Vectors**.

| Component | Content | Shape | Description |
| :--- | :--- | :--- | :--- |
| **Points** | `mesh.points` | $[N, 3]$ | **Reference Geometry.** The node coordinates at $t=0$ (Undisplaced). |
| **Cells** | `mesh.faces` | $[M, 3/4]$ | **Connectivity.** Triangles and Quads used for rendering. |
| **Point Data** | `thickness` | $[N]$ | Physical thickness property. |
| **Point Data** | `displacement_tX.XXX` | $[N, 3]$ | **One array per timestep.**<br>e.g., `displacement_t0.005`, `displacement_t0.010`. |
| **Field Data** | `bc_velocity_vector` | $(3,)$ | **Boundary Condition.** Impact velocity vector. |
| **Field Data** | `bc_rwall_diameter` | $(1,)$ | **Boundary Condition.** Rigid wall cylinder diameter. |
| **Field Data** | `bc_rwall_origin` | $(3,)$ | **Boundary Condition.** Rigid wall center position. |

</div></div>

### **C. Running the VTP Extraction**
To switch formats, we simply change `serialization_format=vtp` in the CLI. The Curator loads the `vtp.yaml` config instead of `zarr.yaml`.


In [ ]:
# Execute Curator with VTP Sink
!python physicsnemo-curator/examples/structural_mechanics/crash/run_etl.py \
    --config-dir=configs \
    --config-name=crash_etl \
    serialization_format=vtp \
    etl.source.input_dir=$FEA_DATA_DIR \
    serialization_format.sink.output_dir=/workspace/processed_data_vtp \
    serialization_format.sink.overwrite_existing=true \
    etl.processing.num_processes=8

Of note, this format **is not** already vendored, so this conversion is net-new for us. Let's quickly check out the structure:

In [ ]:
from IPython.display import Code
with open("/workspace/processed_data_vtp/run2.vtp", "r") as f:
    vtp_body = f.read()
Code(vtp_body[:2000] + "\n\t...\n" + vtp_body[-500:], language="xml")

### **D. Visualization: Inspecting Physics & Geometry**

The following code block visualizes the generated VTP data to ensure the physical fields are correct before training. We perform three key checks:

---

### **Visualization of VTP Results**

In [ ]:
import importlib
import utils
from utils.plotting_utils import load_vtp_data, print_vtp_summary, plot_vtp_multifield

run_name = "run1"
VTP_FILE_PATH = Path.cwd() / "processed_data_vtp" / f"{run_name}.zarr"
SUBSAMPLE_STEP = 1  # Increase for faster rendering (less points)

# ===========================================

# Load, summarize, and visualize VTP data
print("=" * 60)
print("Loading VTP file...")
vtp_data = load_vtp_data(VTP_FILE_PATH)

print_vtp_summary(vtp_data)
plot_vtp_multifield(vtp_data, subsample_step=SUBSAMPLE_STEP)

## **11. Conclusion**

We have successfully processed the raw **FEA Crash Simulation** data and converted it into a **Zarr** format optimized for high-performance ML training.

### Key Takeaways
* **Built-in Domain Expertise:** [**PhysicsNeMo-Curator**](https://github.com/NVIDIA/physicsnemo-curator/tree/main) provides specialized readers and transforms designed specifically for physics data.

| Domain | Formats | Typical Use Case |
|--------|---------|------------------|
| **CFD / External Aerodynamics** | STL, VTU, VTP | Automotive aero (DrivAerML, AhmedML, etc.) |
| **Structural Mechanics / Crash** | D3PLOT, .k (LS-DYNA), .rad (RADIOSS) | Vehicle crash simulation |
| **Thermal / Structural FEA** | Ansys RST | Ansys Mechanical thermal analysis |
| **General Scientific** | HDF5 | Any physics simulation stored in HDF5 |


* **Robust Orchestration:** Our configuration-driven approach ensures that ETL pipelines are both repeatable and easy to audit.
* **Seamless Integration:** The output contracts are "ML-ready," aligning perfectly with [**PhysicsNeMo**](https://github.com/NVIDIA/physicsnemo) training pipelines.
* **High-Performance Scaling:** Multi-process, file-level parallelism ensures that even massive datasets are processed efficiently.

### **Next Steps for Developers**

Ready to onboard a new use case? The **PhysicsNeMo-Curator** is built on a modular, plug-and-play architecture designed for rapid scaling across physics domains. 

To get started with a new dataset, follow these steps:

1.  **Define Your Logic:** Ensure your specific **Source**, **Transform**, and **Sink** classes are implemented to handle your data requirements. You can use existing classes as templates:
    * `Source`: See `CrashD3PlotDataSource`
    * `Transform`: See `CrashDataTransformation`
    * `Sink`: See `CrashZarrDataSource`
2.  **Explore Built-ins:** Refer to the [**`examples`**](https://github.com/NVIDIA/physicsnemo-curator/tree/main/examples) directory. You will find a library of pre-built domain-specific readers and transformation classes that you can reuse or extend.
3.  **Configure:** Reference your chosen classes in your **Hydra YAML** configuration file to define the pipeline behavior.
4.  **Execute:** Run the curator to generate your new ML-ready dataset.

**We are now ready for training!**